# OmniVoice Shorts Studio — batch voice generation

Open in Google Colab, choose **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom. Put your authorized reference voice audio in Google Drive, paste your scripts below, and run generation. WAV outputs are saved to `MyDrive/OmniVoiceShorts/outputs`.

> Colab sessions are temporary. Each new session has to install/load the model again; generated files remain in Drive. Use only a voice you own or have permission to clone.

In [ ]:
!pip -q install omnivoice

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import soundfile as sf
from IPython.display import Audio, display
from omnivoice import OmniVoice

# Change this to the actual reference audio file in your Drive (3–10 seconds recommended).
REF_AUDIO_PATH = '/content/drive/MyDrive/OmniVoiceShorts/reference.wav'
OUTPUT_DIR = Path('/content/drive/MyDrive/OmniVoiceShorts/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paste the exact words spoken in your reference clip if you know them.
# Leaving this blank lets the model try automatic transcription (uses extra model resources).
REF_TEXT = ''

if not Path(REF_AUDIO_PATH).is_file():
    raise FileNotFoundError(f'Reference audio not found: {REF_AUDIO_PATH}. Upload it to that Drive location or edit REF_AUDIO_PATH.')

print('Loading OmniVoice model; this can take a while on the first run...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map='cuda:0',
    dtype=__import__('torch').float16,
    load_asr=not bool(REF_TEXT.strip()),
)
print('Model ready.')

## Paste your Shorts scripts
Add or remove items in the list. Each item becomes a separate WAV file. Keep each script as one string.

In [ ]:
SCRIPTS = [
    'Paste your first short script here.',
    'Paste your second short script here.',
    # 'Paste another script here.',
]

# Optional: set a language such as 'English' or 'Urdu'; leave None for automatic detection.
LANGUAGE = None

In [ ]:
if not SCRIPTS or any(not text.strip() for text in SCRIPTS):
    raise ValueError('Add at least one non-empty script to SCRIPTS.')

for i, text in enumerate(SCRIPTS, start=1):
    print(f'Generating {i}/{len(SCRIPTS)}...')
    audio = model.generate(
        text=text.strip(),
        language=LANGUAGE,
        ref_audio=REF_AUDIO_PATH,
        ref_text=REF_TEXT.strip() or None,
    )
    output_path = OUTPUT_DIR / f'short_{i:02d}.wav'
    sf.write(str(output_path), audio[0], model.sampling_rate)
    print(f'Saved: {output_path}')

print('All done. Your WAV files are in Google Drive → MyDrive → OmniVoiceShorts → outputs.')

## Listen to the latest file (optional)
Run this after generation if you want to preview the last script.

In [ ]:
latest = OUTPUT_DIR / f'short_{len(SCRIPTS):02d}.wav'
display(Audio(filename=str(latest)))